# 🚀 BOT TỰ ĐỘNG KÉO 1000 TẬP ANIME SẠCH VÀO GOOGLE DRIVE 5TB
**Tính năng:**
- Tự động tìm kiếm và lấy 1000 tập anime mới nhất, chất lượng 1080p từ SubsPlease/Erai-raws (Nyaa RSS).
- Tải đa luồng bằng đường truyền 1Gbps của Google Colab.
- Lưu tự động vào thư mục `KhangFlix_Anime` trên Drive 5TB.
- Tự động quét Google Drive ID và tạo sẵn file `data_drive.json` cho app KhangFlix!

In [ ]:
# 1. Kết nối Google Drive 5TB
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/KhangFlix_Anime'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Cài đặt thư viện cần thiết
!apt-get install -y python3-libtorrent
!pip install feedparser google-api-python-client
print("\n[OK] Môi trường và Drive 5TB đã sẵn sàng!")

In [ ]:
# 2. Thu thập 1000 link anime sạch từ RSS Nyaa/SubsPlease
import feedparser
import urllib.parse

print("[*] Đang cào danh sách 1000 tập anime 1080p sạch...")
magnets_queue = []

# Cào từ RSS feed của SubsPlease và Erai-raws (Nguồn anime sạch số 1 thế giới)
feeds = [
    'https://nyaa.si/?page=rss&q=SubsPlease+1080p&c=1_2&f=0',
    'https://nyaa.si/?page=rss&q=Erai-raws+1080p&c=1_2&f=0'
]

for feed_url in feeds:
    d = feedparser.parse(feed_url)
    for entry in d.entries:
        title = entry.title
        link = entry.link
        # Nếu là link torrent hoặc magnet
        magnets_queue.append({'title': title, 'link': link})
        if len(magnets_queue) >= 1000:
            break
    if len(magnets_queue) >= 1000:
        break

print(f"[+] Đã lấy được {len(magnets_queue)} tập anime sạch vào hàng đợi!")

In [ ]:
# 3. Tiến trình tải tự động 1000 tập vào Drive 5TB
import libtorrent as lt
import time, sys

ses = lt.session()
ses.listen_on(6881, 6891)
ses.start_dht()

total = len(magnets_queue)
print(f"[*] Bắt đầu tải {total} tập anime vào: {DRIVE_DIR}\n")

for idx, item in enumerate(magnets_queue, 1):
    print(f"\n[{idx}/{total}] Đang tải: {item['title'][:60]}...")
    params = {
        'save_path': DRIVE_DIR,
        'storage_mode': lt.storage_mode_t(2)
    }
    try:
        handle = lt.add_magnet_uri(ses, item['link'], params)
        # Đợi metadata
        timeout = 20
        while not handle.has_metadata() and timeout > 0:
            time.sleep(1)
            timeout -= 1
            
        if not handle.has_metadata():
            print("  -> Bỏ qua (hết thời gian chờ seeder)")
            ses.remove_torrent(handle)
            continue
            
        # Theo dõi tiến độ
        while handle.status().state != lt.torrent_status.seeding and handle.status().state != lt.torrent_status.finished:
            s = handle.status()
            print(f"\r  Tiến độ: {s.progress * 100:.1f}% | Tốc độ: {s.download_rate / 1000000:.1f} MB/s | Peers: {s.num_peers}", end="")
            sys.stdout.flush()
            time.sleep(3)
        print("\n  -> [Xong] Đã lưu vào Drive!")
        ses.remove_torrent(handle)
    except Exception as e:
        print(f"  -> Lỗi: {e}")

print("\n\n🎉 ĐÃ HOÀN TẤT TẢI TOÀN BỘ ANIME VÀO DRIVE 5TB!")

In [ ]:
# 4. Quét toàn bộ Drive xuất JSON tự nạp vào KhangFlix
from googleapiclient.discovery import build
from google.colab import auth
import json

auth.authenticate_user()
drive_service = build('drive', 'v3')

print("[*] Đang tạo danh sách anime JSON từ Google Drive 5TB...")
page_token = None
drive_episodes = []

while True:
    response = drive_service.files().list(
        q="mimeType contains 'video/' and trashed = false",
        fields="nextPageToken, files(id, name, size)",
        pageSize=500,
        pageToken=page_token
    ).execute()
    for file in response.get('files', []):
        drive_episodes.append({
            "name": file['name'],
            "driveId": file['id'],
            "url": "",
            "archiveId": "",
            "file": ""
        })
    page_token = response.get('nextPageToken', None)
    if page_token is None:
        break

with open('/content/drive/MyDrive/KhangFlix_Anime/catalog_drive.json', 'w', encoding='utf-8') as f:
    json.dump(drive_episodes, f, ensure_ascii=False, indent=2)

print(f"[OK] Đã quét xong {len(drive_episodes)} tập! Đã lưu file catalog_drive.json trên Drive.")